<div style="background-color:#EAF4EC; padding:20px; border-radius:10px;">

<h2 style="color:#2F6F4E; text-align:center; margin-bottom:5px;">
Forecasting: Random Forest
</h2>

<h4 style="color:#2F6F4E; text-align:center; margin-top:0;">
Master Thesis – ESG Governance Indicators (EU-27)
</h4>

<p style="font-size:14px; color:#2F6F4E;">
This notebook implements the <strong>Random Forest forecasting model</strong> as part of the CRISP-ML(Q) methodology.
The objective is to evaluate the predictive performance of a <strong>non-linear ensemble learning approach</strong>
based on bagged decision trees for forecasting ESG governance indicators across the EU-27.
</p>

<p style="font-size:14px; color:#2F6F4E;">
Random Forest models are particularly suitable for capturing <strong>complex non-linear relationships and
variable interactions</strong> while maintaining strong generalization performance through bootstrap aggregation
and randomized feature selection.
</p>

<p style="font-size:14px; color:#2F6F4E;">
The model uses the same set of engineered predictors—lagged values, rolling statistics, and trend-based
features—to ensure comparability with the econometric baseline and other machine learning models.
Forecasting performance is assessed using <strong>out-of-sample evaluation metrics</strong> (MAE, RMSE, and R²),
allowing a systematic comparison with the fixed effects panel regression model and alternative
machine learning approaches.
</p>

</div>

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Import correct dataset

</div>

In [2]:
DATA_PATH = Path("../data/processed/data_final_governance_EU27.csv")
df_final = pd.read_csv(DATA_PATH)
print(df_final.shape)
df_final.head(1)

(405, 28)


,Country Name,Country Code,Indicator Name,Indicator Code,2000,2001,2002,2003,2004,2005,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,1.751059,1.751059,1.915159,1.963869,2.026624,1.912423,...,1.466502,1.472044,1.496881,1.50082,1.568587,1.521324,1.477709,1.242727,1.258587,1.133653


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Use the best Random Forest Hyperparameters acording to Optuna

</div>

In [3]:
rf_parameters = {
    "n_estimators": 538,
    "max_depth": 5,
    "min_samples_split": 6,
    "min_samples_leaf": 4,
    "max_features": 0.8540989975527579,
    "bootstrap": True,
    "random_state": 42,
    "n_jobs": -1
}

ROLL = 3
TRAIN_END = 2018
VAL_END = 2021
FORECAST_START = 2024
FORECAST_END = 2030

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Convert df_final in df_long 

</div>

In [4]:
df = df_final.copy()

# detectar colunas ano (strings tipo "2000", "2001", ...)
year_cols = [c for c in df.columns if str(c).isdigit()]
id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

df_long = df.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

df_long["year"] = pd.to_numeric(df_long["year"], errors="coerce").astype(int)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

# manter só 2000–2023 (observado)
df_long = df_long[df_long["year"].between(2000, 2023)].copy()

# remover linhas sem valor
df_long = df_long.dropna(subset=["value"]).copy()

df_long.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,value
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Define the metrics and temporal features 

</div>

In [5]:
def eval_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [6]:
FEATURES = ["year","lag_1","lag_2","lag_3","rolling_mean_3","rolling_std_3","trend"]

def add_time_features(df_in, roll=3):
    df = df_in.sort_values(["Country Code", "year"]).copy()

    # lags
    for lag in [1, 2, 3]:
        df[f"lag_{lag}"] = df.groupby("Country Code")["value"].shift(lag)

    # rolling (past only)
    g = df.groupby("Country Code")["value"]
    df["rolling_mean_3"] = g.transform(lambda s: s.shift(1).rolling(roll).mean())
    df["rolling_std_3"]  = g.transform(lambda s: s.shift(1).rolling(roll).std())

    # trend
    df["trend"] = df["lag_1"] - df["lag_2"]

    return df


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Train and evaluate model (1 indicator at the time)

</div>

In [7]:
indicator_codes = sorted(df_long["Indicator Code"].unique())
results = []
models  = {}

for code in indicator_codes:
    df_ind = df_long[df_long["Indicator Code"] == code].copy()
    if df_ind.empty:
        continue

    ind_name = df_ind["Indicator Name"].iloc[0]
    df_ind   = add_time_features(df_ind, roll=ROLL)

    req      = FEATURES + ["value"]
    df_model = df_ind.dropna(subset=req).copy()
    if df_model.empty:
        continue

    train_df = df_model[df_model["year"] <= TRAIN_END].copy()
    val_df   = df_model[(df_model["year"] > TRAIN_END) & (df_model["year"] <= VAL_END)].copy()
    test_df  = df_model[df_model["year"] > VAL_END].copy()

    if train_df.empty or val_df.empty or test_df.empty:
        continue

    model = RandomForestRegressor(**rf_parameters)  # ← única diferença
    model.fit(train_df[FEATURES], train_df["value"])

    val_pred  = model.predict(val_df[FEATURES])
    test_pred = model.predict(test_df[FEATURES])

    mae_val,  rmse_val,  r2_val  = eval_metrics(val_df["value"],  val_pred)
    mae_test, rmse_test, r2_test = eval_metrics(test_df["value"], test_pred)

    models[code] = model
    results.append({
        "Indicator Code": code,
        "MAE_Val":    mae_val,
        "RMSE_Val":   rmse_val,
        "R2_Val":     r2_val,
        "MAE_Test":   mae_test,
        "RMSE_Test":  rmse_test,
        "R2_Test":    r2_test,
        "Train_Years": f"{train_df['year'].min()}-{train_df['year'].max()}",
        "Val_Years":   f"{val_df['year'].min()}-{val_df['year'].max()}",
        "Test_Years":  f"{test_df['year'].min()}-{test_df['year'].max()}",
        "N_Train":   len(train_df),
        "N_Val":     len(val_df),
        "N_Test":    len(test_df),
        "Countries": df_ind["Country Code"].nunique()
    })
    print(f" {code} | MAE_Test: {mae_test:.4f} | RMSE_Test: {rmse_test:.4f} | R2_Test: {r2_test:.4f}")
  

 CC.EST | MAE_Test: 0.0593 | RMSE_Test: 0.0734 | R2_Test: 0.9904
 GB.XPD.RSDV.GD.ZS | MAE_Test: 0.0608 | RMSE_Test: 0.0791 | R2_Test: 0.9923
 GE.EST | MAE_Test: 0.1000 | RMSE_Test: 0.1334 | R2_Test: 0.9437
 IP.JRN.ARTC.SC | MAE_Test: 1557.6087 | RMSE_Test: 2767.5787 | R2_Test: 0.9906
 IP.PAT.RESD | MAE_Test: 414.5714 | RMSE_Test: 1505.4202 | R2_Test: 0.9631
 IT.NET.USER.ZS | MAE_Test: 1.3202 | RMSE_Test: 1.9252 | R2_Test: 0.8444
 NY.GDP.MKTP.KD.ZG | MAE_Test: 2.1752 | RMSE_Test: 2.8075 | R2_Test: -0.0637
 PV.EST | MAE_Test: 0.0768 | RMSE_Test: 0.0943 | R2_Test: 0.8097
 RL.EST | MAE_Test: 0.0494 | RMSE_Test: 0.0664 | R2_Test: 0.9861
 RQ.EST | MAE_Test: 0.0691 | RMSE_Test: 0.0864 | R2_Test: 0.9701
 SE.ENR.PRSC.FM.ZS | MAE_Test: 0.0042 | RMSE_Test: 0.0058 | R2_Test: 0.9560
 SG.GEN.PARL.ZS | MAE_Test: 1.4758 | RMSE_Test: 2.7506 | R2_Test: 0.9111
 SL.TLF.CACT.FM.ZS | MAE_Test: 0.7535 | RMSE_Test: 0.9702 | R2_Test: 0.9679
 SM.POP.NETM | MAE_Test: 102075.1628 | RMSE_Test: 203973.6749 | R2_Tes

In [8]:
results_rf = pd.DataFrame(results).sort_values("RMSE_Test").reset_index(drop=True)
results_rf

,Indicator Code,MAE_Val,RMSE_Val,R2_Val,MAE_Test,RMSE_Test,R2_Test,Train_Years,Val_Years,Test_Years,N_Train,N_Val,N_Test,Countries
0,SE.ENR.PRSC.FM.ZS,0.003999,0.006048,0.951321,0.004188,0.005798,0.956005,2003-2018,2019-2021,2022-2023,432,81,54,27
1,VA.EST,0.047627,0.064833,0.966854,0.049203,0.061301,0.972353,2003-2018,2019-2021,2022-2023,432,81,54,27
2,RL.EST,0.052687,0.066852,0.986426,0.049360,0.066413,0.986082,2003-2018,2019-2021,2022-2023,432,81,54,27
3,CC.EST,0.083207,0.115018,0.977114,0.059262,0.073368,0.990360,2003-2018,2019-2021,2022-2023,432,81,54,27
4,GB.XPD.RSDV.GD.ZS,0.080943,0.109838,0.984729,0.060779,0.079112,0.992254,2003-2018,2019-2021,2022-2023,432,81,54,27
5,RQ.EST,0.101718,0.137961,0.913183,0.069062,0.086446,0.970139,2003-2018,2019-2021,2022-2023,432,81,54,27
6,PV.EST,0.081200,0.105428,0.839085,0.076794,0.094289,0.809663,2003-2018,2019-2021,2022-2023,432,81,54,27
7,GE.EST,0.079385,0.101494,0.967921,0.099996,0.133377,0.943663,2003-2018,2019-2021,2022-2023,432,81,54,27
8,SL.TLF.CACT.FM.ZS,0.808410,1.074772,0.961456,0.753541,0.970200,0.967911,2003-2018,2019-2021,2022-2023,432,81,54,27
9,IT.NET.USER.ZS,1.672396,2.145905,0.912451,1.320186,1.925176,0.844365,2003-2018,2019-2021,2022-2023,432,81,54,27


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Recurcive Forecast: 2024 to 2030

</div>

In [9]:
forecast_rows = []

for code in indicator_codes:
    if code not in models:
        continue

    model = models[code]
    df_ind = df_long[df_long["Indicator Code"] == code].copy()
    if df_ind.empty:
        continue

    ind_name = df_ind["Indicator Name"].iloc[0]

    for (ccode, cname), g in df_ind.groupby(["Country Code", "Country Name"]):
        g = g.sort_values("year").copy()

        # RECOMENDADO: usar só observado até 2023 como histórico
        g = g[g["year"] <= 2023].copy()

        # precisamos de pelo menos 4 valores para lag_1..lag_3
        if len(g) < 4:
            continue

        hist_years = g["year"].tolist()
        hist_vals = g["value"].astype(float).tolist()

        last_year = int(max(hist_years))
        start_year = max(FORECAST_START, last_year + 1)

        # se já não há nada para prever
        if start_year > FORECAST_END:
            continue

        # estender a série com previsões (recursivo)
        hist_extended = hist_vals.copy()

        for y in range(start_year, FORECAST_END + 1):
            lag_1 = hist_extended[-1]
            lag_2 = hist_extended[-2]
            lag_3 = hist_extended[-3]

            last3 = hist_extended[-ROLL:]
            rolling_mean_3 = float(np.mean(last3))
            rolling_std_3  = float(np.std(last3, ddof=1)) if len(last3) >= 2 else 0.0
            trend = lag_1 - lag_2

            X_next = pd.DataFrame([{
                "year": int(y),
                "lag_1": float(lag_1),
                "lag_2": float(lag_2),
                "lag_3": float(lag_3),
                "rolling_mean_3": rolling_mean_3,
                "rolling_std_3": rolling_std_3,
                "trend": float(trend),
            }])

            y_hat = float(model.predict(X_next[FEATURES])[0])
            hist_extended.append(y_hat)

            forecast_rows.append({
                "Country Name": cname,
                "Country Code": ccode,
                "Indicator Name": ind_name,
                "Indicator Code": code,
                "year": int(y),
                "pred_rf": y_hat,      
                "type": "forecast"
            })

df_forecast_rf = pd.DataFrame(forecast_rows)
df_forecast_rf.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,pred_rf,type
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2024,1.179853,forecast


In [10]:
# Observed (2000–2023)  
df_observed_rf = df_long.copy()

# garantir que observed é mesmo só até 2023
df_observed_rf = df_observed_rf[df_observed_rf["year"].between(2000, 2023)].copy()

# rename value -> pred_rf
df_observed_rf = df_observed_rf.rename(columns={"value": "pred_rf"})
df_observed_rf["type"] = "observed"

df_observed_rf = df_observed_rf[[
    "Country Name","Country Code","Indicator Name","Indicator Code","year","pred_rf","type"
]]

# Forecast (2024–2030)
df_forecast_rf = df_forecast_rf.copy()
df_forecast_rf = df_forecast_rf[df_forecast_rf["year"].between(2024, 2030)].copy()

df_forecast_rf = df_forecast_rf[[
    "Country Name","Country Code","Indicator Name","Indicator Code","year","pred_rf","type"
]]


# Merge full panel 
df_rf_full = pd.concat([df_observed_rf, df_forecast_rf], ignore_index=True)

#  regra final: observed 2000–2023, forecast 2024–2030
df_rf_full = df_rf_full[
    ((df_rf_full["type"] == "observed") & (df_rf_full["year"].between(2000, 2023))) |
    ((df_rf_full["type"] == "forecast") & (df_rf_full["year"].between(2024, 2030)))
].copy()

# zero duplicados por (country, indicator, year)
# keep="last" -> se houver conflito, fica com forecast (porque "forecast" vem depois no concat)
df_rf_full = (
    df_rf_full.sort_values(["Country Code","Indicator Code","year","type"])
    .drop_duplicates(subset=["Country Code","Indicator Code","year"], keep="last")
    .reset_index(drop=True)
)

df_rf_full.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,pred_rf,type
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059,observed


In [11]:
print("Years observed:",
      df_rf_full[df_rf_full["type"]=="observed"]["year"].min(),
      "-",
      df_rf_full[df_rf_full["type"]=="observed"]["year"].max())

print("Years forecast:",
      df_rf_full[df_rf_full["type"]=="forecast"]["year"].min(),
      "-",
      df_rf_full[df_rf_full["type"]=="forecast"]["year"].max())

dups = df_rf_full.duplicated(
    subset=["Country Code","Indicator Code","year"]
).sum()

print("Duplicates:", dups)

print("Forecast coverage in 2030 (countries per indicator):")
print(
    df_rf_full[df_rf_full["year"]==2030]
    .groupby("Indicator Code")["Country Code"]
    .nunique()
    .sort_values()
)

Years observed: 2000 - 2023
Years forecast: 2024 - 2030
Duplicates: 0
Forecast coverage in 2030 (countries per indicator):
Indicator Code
CC.EST               27
GB.XPD.RSDV.GD.ZS    27
GE.EST               27
IP.JRN.ARTC.SC       27
IP.PAT.RESD          27
IT.NET.USER.ZS       27
NY.GDP.MKTP.KD.ZG    27
PV.EST               27
RL.EST               27
RQ.EST               27
SE.ENR.PRSC.FM.ZS    27
SG.GEN.PARL.ZS       27
SL.TLF.CACT.FM.ZS    27
SM.POP.NETM          27
VA.EST               27
Name: Country Code, dtype: int64


## Random Forest Results — Interpretation

### Group 1 — Excellent (R²_Test > 0.90)

| Indicator Code | RMSE_Test | R²_Test |
|---|---:|---:|
| SE.ENR.PRSC.FM.ZS | 0.0058 | **0.9560** |
| VA.EST | 0.0613 | **0.9724** |
| RL.EST | 0.0664 | **0.9861** |
| CC.EST | 0.0734 | **0.9904** |
| GB.XPD.RSDV.GD.ZS | 0.0791 | **0.9923** |
| RQ.EST | 0.0864 | **0.9701** |
| GE.EST | 0.1334 | **0.9437** |
| SG.GEN.PARL.ZS | 2.7506 | **0.9111** |
| IP.PAT.RESD | 1505.4202 | **0.9631** |
| IP.JRN.ARTC.SC | 2767.5787 | **0.9906** |

Random Forest delivers excellent predictive performance for most indicators, particularly institutional and persistent series, with R² above 0.94 in nearly all cases.

---

### Group 2 — Good (R²_Test 0.75–0.90)

| Indicator Code | RMSE_Test | R²_Test |
|---|---:|---:|
| PV.EST | 0.0943 | **0.8097** |
| SL.TLF.CACT.FM.ZS | 0.9702 | **0.9679** |
| IT.NET.USER.ZS | 1.9252 | **0.8444** |

Performance remains good for a smaller set of indicators, suggesting that Random Forest captures relevant temporal patterns even when predictive fit is lower than in the strongest cases.

---

### Group 3 — Poor (R²_Test < 0)

| Indicator Code | RMSE_Test | R²_Test |
|---|---:|---:|
| NY.GDP.MKTP.KD.ZG | 2.8075 | **-0.0637** |
| SM.POP.NETM | 203973.6749 | **-0.0013** |

GDP growth and net migration remain difficult to forecast. Negative R² indicates that Random Forest does not outperform a simple mean benchmark for these highly volatile indicators.

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Save data as .csv

</div>

In [12]:
out_path = "../../data/processed/RF_2000_2030.csv"
df_rf_full.to_csv(out_path, index=False)
print("Saved:", out_path, "| shape:", df_rf_full.shape)

results_rf.to_csv("../../data/processed/metrics_rf.csv", index=False)

Saved: ../../data/processed/RF_2000_2030.csv | shape: (12555, 7)
